# pdex ↔ cell-eval fold-change double-log bug

This notebook demonstrates a bug in how `cell-eval` interprets the `fold_change` column returned by `pdex`.

**The claim:**
- `pdex` returns `fold_change = log2(target_mean / ref_mean)` — already log2 fold change.
- `cell_eval._types._de.DEResults.__post_init__` then applies `log2` **again** to build its `log2_fold_change` column, producing `log2(log2(ratio))`.
- Negative inputs to `log2` become `NaN`, which cell-eval then silently `fill_nan(0.0)`s — destroying signal.

Run the cells below to see it.

## 1. Imports & synthetic data

We build a small random AnnData with a control and three perturbations using cell-eval's own test utility. `normlog=True` returns log1p-normalised data, matching what cell-eval passes to pdex in production.

In [1]:
import numpy as np
import polars as pl
from pdex import pdex

from cell_eval.data import build_random_anndata
from cell_eval._types._de import initialize_de_comparison
from cell_eval._types._enums import DESortBy

adata = build_random_anndata(
    n_cells=600,
    n_genes=50,
    n_perts=3,
    pert_col="target",
    control_var="non-targeting",
    normlog=True,
)
adata

AnnData object with n_obs × n_vars = 600 × 50
    obs: 'target', 'celltype'

## 2. Run pdex directly

These are the exact kwargs `state tx predict` → `MetricsEvaluator` passes through.

In [7]:
de = pdex(
    adata=adata,
    mode="ref",
    reference="non-targeting",
    groupby="target",
    is_log1p=True,
    threads=1,
)
de.head()

Running parallel differential expression (against reference): 100%|██████████| 4/4 [00:03<00:00,  1.26it/s]


target,feature,target_mean,ref_mean,target_membership,ref_membership,fold_change,percent_change,p_value,statistic,fdr
str,str,f64,f64,i32,i32,f64,f64,f64,f64,f64
"""pert_0""","""0""",145.459302,155.422811,155,144,-0.095583,-0.064106,0.237975,10278.0,0.841031
"""pert_0""","""1""",158.841014,138.127576,155,144,0.201582,0.149959,0.34153,11871.0,0.841031
"""pert_0""","""2""",146.735216,135.163202,155,144,0.118513,0.085615,0.091264,12422.0,0.74588
"""pert_0""","""3""",142.730982,138.453709,155,144,0.043895,0.030893,0.466869,11704.0,0.841031
"""pert_0""","""4""",163.416659,120.275979,155,144,0.442207,0.358681,0.074236,12494.0,0.74588


### What is `fold_change`?

pdex's source (`pdex/_math.py`) defines:

```python
@nb.njit(parallel=True)
def fold_change(x, y, epsilon=0.0):
    return np.log2((x + epsilon) / (y + epsilon))
```

So `fold_change` is **already log2 fold change**. We can sanity-check that by comparing to `log2(target_mean / ref_mean)`:

In [8]:
check = de.with_columns(
    recomputed_log2fc=(pl.col("target_mean") / pl.col("ref_mean")).log(base=2)
).select(["target", "feature", "target_mean", "ref_mean", "fold_change", "recomputed_log2fc"])
check.head(5)

target,feature,target_mean,ref_mean,fold_change,recomputed_log2fc
str,str,f64,f64,f64,f64
"""pert_0""","""0""",145.459302,155.422811,-0.095583,-0.095583
"""pert_0""","""1""",158.841014,138.127576,0.201582,0.201582
"""pert_0""","""2""",146.735216,135.163202,0.118513,0.118513
"""pert_0""","""3""",142.730982,138.453709,0.043895,0.043895
"""pert_0""","""4""",163.416659,120.275979,0.442207,0.442207


In [9]:
fc = de["fold_change"].to_numpy()
print(f"fold_change range: [{fc.min():.4f}, {fc.max():.4f}]")
print(f"#negative: {(fc < 0).sum()} / {fc.size}")
print(f"#|fc| < 1: {(np.abs(fc) < 1).sum()} / {fc.size}")
print("\nIf these were RAW fold changes you would expect positive values only,")
print("centred around 1. Negatives and values<1 confirm they are log2FC.")

fold_change range: [-0.6076, 0.4422]
#negative: 73 / 150
#|fc| < 1: 150 / 150

If these were RAW fold changes you would expect positive values only,
centred around 1. Negatives and values<1 confirm they are log2FC.


## 3. Pass through cell-eval's `DEResults`

`initialize_de_comparison` wraps the pdex frame in a `DEResults`, whose `__post_init__` adds a `log2_fold_change` column via `pl.col(fold_change).log(base=2)`. This is the buggy step.

In [10]:
cmp = initialize_de_comparison(real=de.clone(), pred=de.clone())
transformed = cmp.real.data
transformed.head()

target,feature,fold_change,p_value,fdr,log2_fold_change,abs_log2_fold_change
cat,cat,f32,f32,f32,f32,f32
"""pert_0""","""0""",-0.095583,0.237975,0.841031,0.0,0.0
"""pert_0""","""1""",0.201582,0.341531,0.841031,-2.31056,2.31056
"""pert_0""","""2""",0.118513,0.091264,0.74588,-3.076886,3.076886
"""pert_0""","""3""",0.043895,0.466869,0.841031,-4.509805,4.509805
"""pert_0""","""4""",0.442207,0.074236,0.74588,-1.177208,1.177208


In [11]:
l2 = transformed["log2_fold_change"].to_numpy()
print(f"cell-eval log2_fold_change range: [{l2.min():.4f}, {l2.max():.4f}]")
print(f"#zeros (NaN from log of negatives, filled with 0): {(l2 == 0).sum()} / {l2.size}")

cell-eval log2_fold_change range: [-9.2997, 0.0000]
#zeros (NaN from log of negatives, filled with 0): 73 / 150


## 4. Side-by-side: see the double-log

Column 1 is pdex's `fold_change` (= true log2FC).
Column 2 is cell-eval's `log2_fold_change` (= log2 applied on top).
Column 3 manually reconstructs `log2(log2_fc)` where `log2_fc > 0`, else 0. It matches column 2 exactly — proving the double-log.

In [12]:
pl.DataFrame({
    "pdex_fold_change (true log2FC)": de["fold_change"],
    "cell_eval_log2_fold_change (double-logged)": transformed["log2_fold_change"],
    "expected log2(log2FC) with 0 for negatives": pl.Series(
        np.where(fc > 0, np.log2(np.clip(fc, 1e-30, None)), 0.0)
    ),
}).head(10)

pdex_fold_change (true log2FC),cell_eval_log2_fold_change (double-logged),expected log2(log2FC) with 0 for negatives
f64,f32,f64
-0.095583,0.0,0.0
0.201582,-2.31056,-2.310561
0.118513,-3.076886,-3.076886
0.043895,-4.509805,-4.509805
0.442207,-1.177208,-1.177208
0.150712,-2.730138,-2.730138
0.13659,-2.872073,-2.872073
0.171684,-2.542172,-2.542172
-0.086533,0.0,0.0


## 5. Ranking impact

The real damage: metrics like `overlap_at_N`, `precision_at_N`, and top-gene selection use `abs_log2_fold_change` for ranking. Because `log2(x)` for `0 < x < 1` grows more negative as `x → 0`, **genes with the smallest real effects get the largest corrupted magnitudes**, and everything with a negative true log2FC is forced to 0.

Compare the ranking by true log2FC against the ranking by cell-eval's corrupted `abs_log2_fold_change`:

In [13]:
pert = "pert_0"

true_rank = (
    de.filter(pl.col("target") == pert)
      .with_columns(abs_true=pl.col("fold_change").abs())
      .sort("abs_true", descending=True)
      .select(["feature", "fold_change", "abs_true"])
      .head(10)
)
print("TRUE top-10 by |log2FC| (using pdex fold_change):")
true_rank

TRUE top-10 by |log2FC| (using pdex fold_change):


feature,fold_change,abs_true
str,f64,f64
"""28""",-0.459908,0.459908
"""4""",0.442207,0.442207
"""35""",0.326675,0.326675
"""12""",0.238976,0.238976
"""37""",-0.219247,0.219247
"""42""",0.213542,0.213542
"""39""",-0.208299,0.208299
"""48""",0.206284,0.206284
"""27""",0.203785,0.203785


In [14]:
corrupt_rank = (
    transformed.filter(pl.col("target") == pert)
               .sort("abs_log2_fold_change", descending=True)
               .select(["feature", "fold_change", "log2_fold_change", "abs_log2_fold_change"])
               .head(10)
)
print("CORRUPTED top-10 by cell-eval abs_log2_fold_change:")
corrupt_rank

CORRUPTED top-10 by cell-eval abs_log2_fold_change:


feature,fold_change,log2_fold_change,abs_log2_fold_change
cat,f32,f32,f32
"""45""",0.024241,-5.366382,5.366382
"""46""",0.038001,-4.717836,4.717836
"""15""",0.04236,-4.561167,4.561167
"""3""",0.043895,-4.509805,4.509805
"""23""",0.07352,-3.765726,3.765726
"""34""",0.07834,-3.674112,3.674112
"""11""",0.107004,-3.224257,3.224257
"""24""",0.107394,-3.21902,3.21902
"""18""",0.114392,-3.127938,3.127938


Notice how the corrupted ranking picks out genes with tiny `fold_change` (e.g. 0.04) because `|log2(0.04)|` is large, while genes with the largest true log2FC sink down the list. The two rankings barely overlap.

## 6. The fix

In `src/cell_eval/_types/_de.py` lines 100–111, replace the double-log with an alias, since pdex's `fold_change` already is log2 fold change:

```python
if self.log2_fold_change_col not in self.data.columns:
    self.data = self.data.with_columns(
        pl.col(self.fold_change_col).alias(self.log2_fold_change_col)
    ).with_columns(
        pl.col(self.log2_fold_change_col).abs().alias(self.abs_log2_fold_change_col)
    )
```